# BERT Sentiment Classification — Baseline vs. Class-Weighted (RQ2)

Runs `src/train.py` + `src/evaluate.py` twice — once with `use_class_weights: false` (baseline) and once with `use_class_weights: true` (RQ2 class-balancing experiment) — then compares the neutral-class F1/recall between the two.

**Before running:** In Colab, go to `Runtime -> Change runtime type` and select a **GPU** hardware accelerator, then `Runtime -> Run all`.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> GPU, then re-run.")

In [ ]:
# Clone the repo and check out the branch with class-weighting support.
# Once PR #8 (https://github.com/Sapna-Patel1/amazon-electronics-sentiment-analysis/pull/8)
# is merged, change BRANCH to "main".
BRANCH = "feat/bert-class-weighting"

!git clone https://github.com/Sapna-Patel1/amazon-electronics-sentiment-analysis.git
%cd amazon-electronics-sentiment-analysis
!git checkout {BRANCH}
!pwd && ls

In [ ]:
!pip install -q -r requirements.txt

## 1. Baseline run (`use_class_weights: false`)

This is already the default in `configs/bert_config.yaml`, but the next cell sets it explicitly so this notebook is safe to re-run from the top.

In [ ]:
import yaml

CONFIG_PATH = "configs/bert_config.yaml"


def set_class_weights(enabled: bool) -> None:
    cfg = yaml.safe_load(open(CONFIG_PATH))
    cfg["training"]["use_class_weights"] = enabled
    yaml.safe_dump(cfg, open(CONFIG_PATH, "w"), sort_keys=False)
    print(f"use_class_weights set to {enabled}")


set_class_weights(False)

In [ ]:
!python src/train.py

In [ ]:
!python src/evaluate.py

## 2. Weighted run (`use_class_weights: true`)

In [ ]:
set_class_weights(True)

In [ ]:
!python src/train.py

In [ ]:
!python src/evaluate.py

## 3. Compare baseline vs. weighted (RQ2 answer)

Reads the structured JSON results both runs saved to `outputs/` and compares them directly — the neutral-class row is the one RQ2 cares about.

In [ ]:
import json

with open("outputs/evaluation_results_baseline.json") as f:
    baseline = json.load(f)

with open("outputs/evaluation_results_weighted.json") as f:
    weighted = json.load(f)

print(f"{'Metric':<20}{'Baseline':>12}{'Weighted':>12}{'Delta':>12}")
print("-" * 56)


def row(label, b, w):
    print(f"{label:<20}{b:>12.4f}{w:>12.4f}{w - b:>+12.4f}")


row("Accuracy", baseline["accuracy"], weighted["accuracy"])
row("F1 Macro", baseline["f1_macro"], weighted["f1_macro"])
row("F1 Weighted", baseline["f1_weighted"], weighted["f1_weighted"])
print()
for sentiment in ["negative", "neutral", "positive"]:
    row(
        f"F1 ({sentiment})",
        baseline["f1_per_class"][sentiment],
        weighted["f1_per_class"][sentiment],
    )

print()
neutral_delta = weighted["f1_per_class"]["neutral"] - baseline["f1_per_class"]["neutral"]
if neutral_delta > 0:
    print(f"RQ2: class weighting IMPROVED neutral-class F1 by {neutral_delta:+.4f}")
elif neutral_delta < 0:
    print(f"RQ2: class weighting HURT neutral-class F1 by {neutral_delta:+.4f}")
else:
    print("RQ2: class weighting made no difference to neutral-class F1")

In [ ]:
import pandas as pd

labels = ["negative", "neutral", "positive"]

print("Baseline confusion matrix (rows=true, cols=predicted):")
display(pd.DataFrame(baseline["confusion_matrix"], index=labels, columns=labels))

print("\nWeighted confusion matrix (rows=true, cols=predicted):")
display(pd.DataFrame(weighted["confusion_matrix"], index=labels, columns=labels))

## 4. (Optional) Push results back to GitHub

Uncomment and run if you want to commit `outputs/` and the trained-model directories back to the repo from this Colab session.

In [ ]:
# from google.colab import userdata
# import subprocess
#
# !git config user.email "you@example.com"
# !git config user.name "Your Name"
# !git add outputs/
# !git commit -m "Add baseline/weighted BERT evaluation results"
# !git push https://<github-token>@github.com/Sapna-Patel1/amazon-electronics-sentiment-analysis.git {BRANCH}